# Optimasi Rute Puskesmas (VRP) - Memetic Algorithm
**Klaster:** All | **Constraint:** 10 Jam Kerja, 20 Menit Pelayanan, & Prioritas Induk Inap

In [1]:
import pandas as pd
import numpy as np
import json
import random
import os
import time

# ==========================================
# 1. PARAMETER GLOBAL & SETUP FOLDER
# ==========================================
DAFTAR_KLASTER = ['Barat', 'Pusat', 'Selatan', 'Timur', 'Utara']

MAX_WORKING_TIME = 600     # 10 jam kerja
SERVICE_TIME = 20          # Waktu layanan puskesmas per titik
PENALTY_VIOLATION = 10000  

# Parameter Algoritma
POP_SIZE = 50
GENERATIONS = 100
CROSSOVER_RATE = 0.85
MUTATION_RATE = 0.15
TOURNAMENT_SIZE = 3
JUMLAH_RUN = 10

folder_output = '../output_json'
os.makedirs(folder_output, exist_ok=True)
file_output_json = f'{folder_output}/rute_ma.json'

# Kerangka utama JSON
database_rute = {"algoritma": "Memetic Algorithm (MA)", "hasil_per_klaster": {}}

print("✅ Setup Global Parameter berhasil dimuat!")

✅ Setup Global Parameter berhasil dimuat!


In [2]:
# ==========================================
# 2. FUNGSI CLEANSING & PRIORITAS
# ==========================================
def bersihkan_latitude(val):
    if pd.isna(val): return 0.0
    val_bersih = str(val).replace(',', '').replace('.', '').replace(' ', '').replace('-', '')
    hasil = val_bersih[:1] + '.' + val_bersih[1:] if len(val_bersih) > 1 else val_bersih
    return float('-' + hasil) if '-' in str(val) else float(hasil)

def bersihkan_longitude(val):
    if pd.isna(val): return 0.0
    val_bersih = str(val).replace(',', '').replace('.', '').replace(' ', '')
    hasil = val_bersih[:3] + '.' + val_bersih[3:] if len(val_bersih) > 3 else val_bersih
    return float(hasil)

def get_priority_tier(nama_lokasi):
    nama_lower = nama_lokasi.lower()
    if 'uptd' in nama_lower or 'gudang farmasi' in nama_lower: return 0
    elif 'inap' in nama_lower: return 1
    elif 'pustu' in nama_lower: return 3
    else: return 2

In [3]:
# ==========================================
# 3. FUNGSI LOGIKA VRP (WAKTU & PRIORITAS)
# ==========================================
def check_priority_violation(route_nodes):
    for i in range(len(route_nodes)):
        for j in range(i + 1, len(route_nodes)):
            if get_priority_tier(route_nodes[i]) > get_priority_tier(route_nodes[j]): return True
    return False

def split_giant_tour(chromosome):
    routes, current_route, current_time, current_dist, prev_node = [], [], 0, 0, depot
    for node in chromosome:
        t_to_node = matriks_waktu.loc[prev_node, node]
        t_to_depot = matriks_waktu.loc[node, depot]
        
        if current_time + t_to_node + SERVICE_TIME + t_to_depot <= MAX_WORKING_TIME:
            current_route.append(node)
            current_time += t_to_node + SERVICE_TIME
            current_dist += matriks_jarak.loc[prev_node, node]
            prev_node = node
        else:
            if current_route:
                routes.append({'route': [depot] + current_route + [depot], 'time': current_time + matriks_waktu.loc[prev_node, depot], 'distance': current_dist + matriks_jarak.loc[prev_node, depot], 'priority_violation': check_priority_violation(current_route)})
            current_route, current_time, current_dist, prev_node = [node], matriks_waktu.loc[depot, node] + SERVICE_TIME, matriks_jarak.loc[depot, node], node
            
    if current_route: routes.append({'route': [depot] + current_route + [depot], 'time': current_time + matriks_waktu.loc[prev_node, depot], 'distance': current_dist + matriks_jarak.loc[prev_node, depot], 'priority_violation': check_priority_violation(current_route)})
    return routes

# 🔥 PERUBAHAN: Sekarang MA menghitung Total Jarak sebagai Fitness
def evaluate_fitness(chromosome):
    routes = split_giant_tour(chromosome)
    return sum(r['distance'] for r in routes) + sum(PENALTY_VIOLATION for r in routes if r['priority_violation'])

In [4]:
# ==========================================
# 4. LOCAL SEARCH (2-OPT) - OPTIMASI JARAK
# ==========================================
def local_search_2opt(route_dict):
    nodes = route_dict['route'][1:-1]
    if len(nodes) < 2: return route_dict
        
    best_dist, best_time, best_nodes, improved = route_dict['distance'], route_dict['time'], nodes.copy(), True
    while improved:
        improved = False
        for i in range(len(best_nodes)):
            for j in range(i + 1, len(best_nodes)):
                new_nodes = best_nodes.copy()
                new_nodes[i:j+1] = list(reversed(new_nodes[i:j+1]))
                if check_priority_violation(new_nodes): continue
                    
                # Hitung waktu baru (hanya untuk cek syarat 10 jam)
                new_time, prev = 0, depot
                for n in new_nodes:
                    new_time += matriks_waktu.loc[prev, n] + SERVICE_TIME
                    prev = n
                new_time += matriks_waktu.loc[prev, depot]
                
                # Hitung jarak baru (untuk optimasi)
                new_dist, prev = 0, depot
                for n in new_nodes:
                    new_dist += matriks_jarak.loc[prev, n]
                    prev = n
                new_dist += matriks_jarak.loc[prev, depot]
                
                # 🔥 PERUBAHAN: 2-Opt sekarang mengutamakan Jarak terpendek
                if new_dist < best_dist and new_time <= MAX_WORKING_TIME:
                    best_dist, best_time, best_nodes, improved = new_dist, new_time, new_nodes, True
                    
    return {'route': [depot] + best_nodes + [depot], 'time': best_time, 'distance': best_dist, 'priority_violation': False}

In [5]:
# ==========================================
# 5. KOMPONEN GENETIC ALGORITHM
# ==========================================
def create_individual():
    t1 = [n for n in daftar_puskesmas if get_priority_tier(n) == 1]
    t2 = [n for n in daftar_puskesmas if get_priority_tier(n) == 2]
    t3 = [n for n in daftar_puskesmas if get_priority_tier(n) == 3]
    random.shuffle(t1); random.shuffle(t2); random.shuffle(t3)
    return t1 + t2 + t3

def tournament_selection(population):
    return min(random.sample(population, TOURNAMENT_SIZE), key=evaluate_fitness)

def ordered_crossover(p1, p2):
    s, e = sorted(random.sample(range(len(p1)), 2))
    child = [None]*len(p1)
    child[s:e+1] = p1[s:e+1]
    ptr = 0
    for i in range(len(p1)):
        if child[i] is None:
            while p2[ptr] in child: ptr += 1
            child[i] = p2[ptr]
    return child

def swap_mutation(chromosome):
    if random.random() < MUTATION_RATE:
        idx1, idx2 = random.sample(range(len(chromosome)), 2)
        chromosome[idx1], chromosome[idx2] = chromosome[idx2], chromosome[idx1]
    return chromosome

In [6]:
# ==========================================
# 6. MAIN ENGINE
# ==========================================
print(f"🚀 MEMULAI PROSES MA VRP ({JUMLAH_RUN}x Run per Klaster)...\n")

for KLASTER in DAFTAR_KLASTER:
    print("="*70)
    print(f"📍 MEMPROSES KLASTER {KLASTER.upper()}")
    print("-" * 70)
    
    # Load Data Spesifik per Klaster
    df_koordinat = pd.read_csv('../data/koordinat_eas.csv')
    matriks_jarak = pd.read_csv(f'../data/matriks_jarak_riil_{KLASTER.lower()}.csv', index_col=0)
    matriks_waktu = pd.read_csv(f'../data/datamatriks_waktu_{KLASTER.lower()}.csv', index_col=0)
    
    dict_koordinat = {row['Nama Puskesmas'].strip(): [bersihkan_latitude(row['Latitude']), bersihkan_longitude(row['Longitude'])] for _, row in df_koordinat.iterrows()}
    depot, daftar_puskesmas = list(matriks_waktu.index)[0], list(matriks_waktu.index)[1:]
    
    # Eksekusi 10 Run
    semua_fitness_akhir, semua_waktu_komputasi = [], []
    global_best_fitness, global_best_chromosome, global_best_konvergensi, global_best_waktu = float('inf'), None, [], 0

    for run in range(JUMLAH_RUN):
        start_time = time.time()
        population = [create_individual() for _ in range(POP_SIZE)]
        riwayat_konvergensi = []
        
        for gen in range(GENERATIONS):
            new_pop = [min(population, key=evaluate_fitness)]
            while len(new_pop) < POP_SIZE:
                p1, p2 = tournament_selection(population), tournament_selection(population)
                child = ordered_crossover(p1, p2) if random.random() < CROSSOVER_RATE else p1.copy()
                child = swap_mutation(child)
                
                opt_tour = []
                for r in split_giant_tour(child): opt_tour.extend(local_search_2opt(r)['route'][1:-1])
                new_pop.append(opt_tour)
                
            population = new_pop
            riwayat_konvergensi.append(evaluate_fitness(min(population, key=evaluate_fitness)))
            
        waktu_run = round(time.time() - start_time, 3)
        semua_waktu_komputasi.append(waktu_run)
        best_run_fitness = evaluate_fitness(min(population, key=evaluate_fitness))
        semua_fitness_akhir.append(best_run_fitness)
        
        # [Visual] Output Run
        print(f"      ➔ Run {run+1}/{JUMLAH_RUN} Selesai | Fitness: {best_run_fitness:.2f} | Waktu: {waktu_run} dtk")
        
        if best_run_fitness < global_best_fitness:
            global_best_fitness, global_best_chromosome, global_best_konvergensi, global_best_waktu = best_run_fitness, min(population, key=evaluate_fitness), riwayat_konvergensi, waktu_run

    # Rekam Data ke Database
    final_sub_routes, total_waktu, total_jarak, rute_json = split_giant_tour(global_best_chromosome), 0, 0, []
    for idx, r in enumerate(final_sub_routes):
        r_opt = local_search_2opt(r)
        total_waktu += r_opt['time']
        total_jarak += r_opt['distance']
        rute_json.append({"id_kurir": idx + 1, "waktu_tempuh_menit": round(r_opt['time'], 2), "jarak_tempuh_km": round(r_opt['distance'], 2), "urutan_kunjungan": r_opt['route'], "koordinat_kunjungan": [dict_koordinat[nama.strip()] for nama in r_opt['route']]})

    fit_min = round(float(np.min(semua_fitness_akhir)), 2)
    fit_mean = round(float(np.mean(semua_fitness_akhir)), 2)
    fit_std = round(float(np.std(semua_fitness_akhir)), 2) 
    runtime_mean = round(float(np.mean(semua_waktu_komputasi)), 3)

    database_rute["hasil_per_klaster"][KLASTER] = {
        "statistik_10_run": {"fitness_minimum": fit_min, "fitness_rata_rata": fit_mean, "fitness_std_dev": fit_std, "waktu_komputasi_rata_rata_detik": runtime_mean, "semua_fitness_run": [round(v, 2) for v in semua_fitness_akhir]},
        "total_kurir": len(final_sub_routes), "waktu_komputasi_detik_terbaik": global_best_waktu, "total_waktu_semua_menit": round(total_waktu, 2), "total_jarak_semua_km": round(total_jarak, 2), "riwayat_konvergensi": [round(v, 2) for v in global_best_konvergensi], "rute_per_kurir": rute_json
    }
    
    # [Visual] Output Summary Klaster
    print("-" * 70)
    print(f"✅ HASIL TERBAIK KLASTER {KLASTER.upper()}:")
    print(f"Total Kurir          : {len(final_sub_routes)} Orang")
    print(f"Fitness Min (Jarak)  : {fit_min} KM")
    print(f"Fitness Rata-Rata    : {fit_mean} KM")
    print(f"Standar Deviasi      : {fit_std}")
    print(f"Waktu Komputasi (Avg): {runtime_mean} Detik")
    print("-" * 70)
    
    # [Visual] Output Rute per Kurir
    for kurir in rute_json:
        print(f"🚚 [KURIR {kurir['id_kurir']}] - Jarak: {kurir['jarak_tempuh_km']} KM | Waktu: {kurir['waktu_tempuh_menit']} Menit")
        rute_singkat = " ➔ ".join([nama.replace("Puskesmas ", "P. ").replace("Pustu ", "P. ") for nama in kurir['urutan_kunjungan']])
        print(f"   Rute: {rute_singkat}\n")

🚀 MEMULAI PROSES MA VRP (10x Run per Klaster)...

📍 MEMPROSES KLASTER BARAT
----------------------------------------------------------------------


      ➔ Run 1/10 Selesai | Fitness: 163.96 | Waktu: 677.641 dtk
      ➔ Run 2/10 Selesai | Fitness: 163.96 | Waktu: 964.298 dtk
      ➔ Run 3/10 Selesai | Fitness: 171.84 | Waktu: 688.295 dtk
      ➔ Run 4/10 Selesai | Fitness: 163.96 | Waktu: 827.614 dtk
      ➔ Run 5/10 Selesai | Fitness: 171.84 | Waktu: 460.179 dtk
      ➔ Run 6/10 Selesai | Fitness: 171.84 | Waktu: 205.612 dtk
      ➔ Run 7/10 Selesai | Fitness: 164.14 | Waktu: 197.266 dtk
      ➔ Run 8/10 Selesai | Fitness: 163.96 | Waktu: 201.292 dtk
      ➔ Run 9/10 Selesai | Fitness: 164.14 | Waktu: 199.136 dtk
      ➔ Run 10/10 Selesai | Fitness: 163.96 | Waktu: 203.924 dtk
----------------------------------------------------------------------
✅ HASIL TERBAIK KLASTER BARAT:
Total Kurir          : 2 Orang
Fitness Min (Jarak)  : 163.96 KM
Fitness Rata-Rata    : 166.36 KM
Standar Deviasi      : 3.59
Waktu Komputasi (Avg): 462.526 Detik
----------------------------------------------------------------------
🚚 [KURIR 1] - Jarak: 130

In [7]:
# ==========================================
# 7. EXPORT FINAL KE JSON
# ==========================================
with open(file_output_json, 'w') as f:
    json.dump(database_rute, f, indent=4)

print("="*70)
print(f"🎉 SEMUA KLASTER SELESAI! Data statistik 10 Run berhasil digabung ke file: {file_output_json}")

🎉 SEMUA KLASTER SELESAI! Data statistik 10 Run berhasil digabung ke file: ../output_json/rute_ma.json
